[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/02_preprocessing.ipynb)

# 02 — Verifying the Data Before Optimizing on It

**Question.** Do the simulation artifacts match their contract, and what exactly may the optimizer change?

**Inputs.** The simulation artifacts, read-only. **Outputs.** `data/processed/mdt.parquet` and `data/processed/cell.parquet`.
The shell equivalent is `task preprocess`. Nothing here is fitted to the data: no imputation, scaling or filtering.

In [1]:
# Environment: locally, move to the project root; on Colab, clone the repository
# and install what Colab lacks. Extra Hydra overrides come from BAND_TILT_OVERRIDES.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    root = Path("/content/band-tilt")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    missing = [pip for module, pip in COLAB_PACKAGES if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
sys.path.insert(0, str(root))
CONFIG_OVERRIDES = os.environ.get("BAND_TILT_OVERRIDES", "").split()

In [2]:
%load_ext autoreload
%autoreload 2

from functools import partial

import pandas as pd

from src.config import load_config
from src.data import schema
from src.data.build import build_cells, build_mdt
from src.data.load import load_artifacts, save
from src.evaluation.export import readable, save_table
from src.utils.plotting import save_fig, setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path("reports/figures/02_preprocessing"))
save_table = partial(
    save_table, in_colab=IN_COLAB, directory=Path("reports/tables/02_preprocessing")
)
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)

## 1. Contract Verification

Every check names the source of the bound it enforces.

In [3]:
artifacts = load_artifacts(cfg)
checks = schema.verify(artifacts, cfg)
checks_table = readable(checks)
save_table(checks_table, "verification_checks")
schema.require(checks)  # raises, listing every failed check
checks_table

,Check,source,Holds,violations
0,npz tx_name matches the configured cells,configs/simulation.yaml,True,0
1,npz band_label matches the configured bands,configs/simulation.yaml,True,0
2,npz scenario_id matches the manifest,scenario.json,True,0
3,npz grid matches the manifest grid,scenario.json,True,0
4,npz ue_height_m matches the config,configs/simulation.yaml,True,0
5,mdt position columns are the declared set,configs/simulation.yaml,True,0
6,"mdt measurement columns are cell x band, in order",radio_map.npz,True,0
7,npz sinr_db has the shape of rsrp_dbm,radio_map.npz,True,0
8,z equals the configured UE height,configs/simulation.yaml,True,0
9,0 <= tile_row < n_rows,scenario.json,True,0


## 2. Decision Variables

One absolute tilt per (cell, band) pair, bounded per band. The optimizer may only move tilts inside these ranges.

In [4]:
cell_table = build_cells(cfg, artifacts)
decision_variables = (
    cell_table.assign(at_upper_bound=cell_table["tilt_baseline_deg"] == cell_table["tilt_max_deg"])
    .groupby("band", observed=True, sort=False)
    .agg(
        cells=("cell", "nunique"),
        baseline=("tilt_baseline_deg", "median"),
        minimum=("tilt_min_deg", "min"),
        maximum=("tilt_max_deg", "max"),
        at_upper_bound=("at_upper_bound", "mean"),
    )
    .reset_index()
    .rename(
        columns={
            "cells": "Cells",
            "baseline": "Current tilt [°]",
            "minimum": "Minimum tilt [°]",
            "maximum": "Maximum tilt [°]",
            "at_upper_bound": "Share of cells at the upper bound",
        }
    )
)
decision_variables = readable(decision_variables)
save_table(decision_variables, "decision_variables")
decision_variables

,Band,Cells,Current tilt [°],Minimum tilt [°],Maximum tilt [°],Share of cells at the upper bound
0,2600 MHz,12,10.0,0.0,10.0,1.0
1,1800 MHz,12,9.0,0.0,10.0,0.0
2,700 MHz,12,8.0,0.0,10.0,0.0


**Observations.** _To be written._

## 3. Processed Tables

The MDT is typed and each measurement gets an explicit `reported_*` flag. No row is dropped and no value changes.

In [5]:
mdt = build_mdt(artifacts)
raw = artifacts.mdt
measurement = [column for column in mdt.columns if column.startswith("rsrp_")]
flags = [column for column in mdt.columns if column.startswith("reported_")]

assert len(mdt) == len(raw), "preprocessing must not drop rows"
assert (mdt[flags].to_numpy() == mdt[measurement].notna().to_numpy()).all()

audit = pd.DataFrame(
    [
        ("Rows", len(raw), len(mdt)),
        ("Columns", raw.shape[1], mdt.shape[1]),
        (
            "Measurements present [%]",
            100 * raw[measurement].notna().to_numpy().mean(),
            100 * mdt[flags].to_numpy().mean(),
        ),
        (
            "Memory [MB]",
            raw.memory_usage(deep=True).sum() / 1e6,
            mdt.memory_usage(deep=True).sum() / 1e6,
        ),
    ],
    columns=["Property", "Simulation output (CSV)", "Processed table (Parquet)"],
)
save_table(audit, "preprocessing_audit")

mdt_path = save(mdt, cfg.data.output.mdt_file)
cell_path = save(cell_table, cfg.data.output.cell_file)
pd.testing.assert_frame_equal(pd.read_parquet(mdt_path), mdt)
pd.testing.assert_frame_equal(pd.read_parquet(cell_path), cell_table)
assert sorted(cell_table["rsrp_column"].astype(str)) == sorted(measurement)
audit

,Property,Simulation output (CSV),Processed table (Parquet)
0,Rows,9929.0000,9929.0000
1,Columns,43.0000,80.0000
2,Measurements present [%],76.1006,76.1006
3,Memory [MB],3.4157,2.0755
